In [0]:
%run /Users/rahulpatel@cyntexa.com/IAP_Practice/freshkart/00_set_up

In [0]:
catalog = dbutils.widgets.get("catalog")

In [0]:
# --- 1. gold.dim_date: one row per day, 2026-03-01 to 2026-03-31 ---
spark.sql(f"""
    CREATE OR REPLACE TABLE {catalog}.gold.dim_date AS
    SELECT
        CAST(date_format(date_add(DATE '2026-03-01', CAST(id AS INT)), 'yyyyMMdd') AS INT) AS date_key,
        date_add(DATE '2026-03-01', CAST(id AS INT))                                   AS calendar_date,
        date_format(date_add(DATE '2026-03-01', CAST(id AS INT)), 'EEEE')               AS day_name,
        CAST(weekofyear(date_add(DATE '2026-03-01', CAST(id AS INT))) AS INT)           AS iso_week,
        dayofweek(date_add(DATE '2026-03-01', CAST(id AS INT))) IN (1, 7)               AS is_weekend
    FROM range(0, 31)
""")

# --- 2. gold.dim_store: one row per store ---
spark.sql(f"""
    CREATE OR REPLACE TABLE {catalog}.gold.dim_store AS
    SELECT store_id, store_name, city, state, region, opening_date
    FROM {catalog}.silver.stores
""")

# --- 3. gold.dim_product: one row per product (exclude tags & last_changed_snapshot_date) ---
spark.sql(f"""
    CREATE OR REPLACE TABLE {catalog}.gold.dim_product AS
    SELECT
        product_id, product_name, brand, category_l1, category_l2,
        packsize, pack_unit, mrp, selling_price, is_active
    FROM {catalog}.silver.products
""")

# --- 4. gold.dim_customer: one row per customer version, with surrogate key ---
spark.sql(f"""
    CREATE OR REPLACE TABLE {catalog}.gold.dim_customer AS
    SELECT
        CAST(ROW_NUMBER() OVER (ORDER BY customer_id, effective_from) AS BIGINT) AS customer_sk,
        customer_id, full_name, email, phone, city, tier,
        signup_date, effective_from, effective_to, is_current
    FROM {catalog}.silver.customers
""")

In [0]:
# ============================================================
# 1. gold.fact_orders  (one row per order, MERGE on order_id)
# ============================================================

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.gold.fact_orders (
        order_id          STRING,
        order_date_key    INT,
        customer_sk       BIGINT,
        customer_id       STRING,
        store_id          STRING,
        order_status      STRING,
        payment_method    STRING,
        channel           STRING,
        line_count        INT,
        units             INT,
        gross_amount      DECIMAL(12,2),
        discount_amount   DECIMAL(12,2),
        net_item_amount   DECIMAL(12,2),
        delivery_fee      DECIMAL(10,2),
        order_total       DECIMAL(12,2),
        promised_mins     INT,
        actual_mins       INT,
        is_on_time        BOOLEAN
    )
""")

spark.sql(f"""
    MERGE INTO {catalog}.gold.fact_orders AS t
    USING (
        SELECT
            o.order_id,
            dd.date_key                                      AS order_date_key,
            dc.customer_sk,
            o.customer_id,
            o.store_id,
            o.order_status,
            o.payment_method,
            o.channel,
            CAST(COALESCE(oa.line_count, 0)      AS INT)         AS line_count,
            CAST(COALESCE(oa.units, 0)            AS INT)         AS units,
            CAST(COALESCE(oa.gross_amount, 0)     AS DECIMAL(12,2)) AS gross_amount,
            CAST(COALESCE(oa.discount_amount, 0)  AS DECIMAL(12,2)) AS discount_amount,
            CAST(COALESCE(oa.net_item_amount, 0)  AS DECIMAL(12,2)) AS net_item_amount,
            o.delivery_fee,
            CAST(COALESCE(oa.net_item_amount, 0) + o.delivery_fee
                 AS DECIMAL(12,2))                                   AS order_total,
            o.promised_mins,
            o.actual_mins,
            CASE
                WHEN o.order_status = 'DELIVERED'
                     AND o.actual_mins <= o.promised_mins THEN TRUE
                WHEN o.order_status = 'DELIVERED'           THEN FALSE
                ELSE NULL
            END                                                AS is_on_time
        FROM {catalog}.silver.orders o
        LEFT JOIN (
            SELECT
                order_id,
                COUNT(*)         AS line_count,
                SUM(quantity)    AS units,
                SUM(gross_amount)    AS gross_amount,
                SUM(discount_amount) AS discount_amount,
                SUM(net_amount)      AS net_item_amount
            FROM {catalog}.silver.order_items
            GROUP BY order_id
        ) oa  ON o.order_id = oa.order_id
        LEFT JOIN {catalog}.gold.dim_date dd
               ON o.order_date = dd.calendar_date
        LEFT JOIN {catalog}.gold.dim_customer dc
               ON o.customer_id = dc.customer_id
              AND o.order_date >= dc.effective_from
              AND (dc.effective_to IS NULL OR o.order_date < dc.effective_to)
    ) AS s
    ON t.order_id = s.order_id
    WHEN MATCHED     THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

# ============================================================
# 2. gold.fact_order_line  (one row per line item, MERGE on order_id + line_number)
# ============================================================

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.gold.fact_order_line (
        order_id          STRING,
        line_number       INT,
        order_date_key    INT,
        customer_sk       BIGINT,
        store_id          STRING,
        product_id        STRING,
        order_status      STRING,
        quantity          INT,
        unit_price        DECIMAL(10,2),
        gross_amount      DECIMAL(12,2),
        discount_amount   DECIMAL(12,2),
        net_amount        DECIMAL(12,2)
    )
""")

spark.sql(f"""
    MERGE INTO {catalog}.gold.fact_order_line AS t
    USING (
        SELECT
            oi.order_id,
            oi.line_number,
            dd.date_key         AS order_date_key,
            dc.customer_sk,
            o.store_id,
            oi.product_id,
            o.order_status,
            oi.quantity,
            oi.unit_price,
            CAST(oi.gross_amount    AS DECIMAL(12,2)) AS gross_amount,
            CAST(oi.discount_amount AS DECIMAL(12,2)) AS discount_amount,
            CAST(oi.net_amount      AS DECIMAL(12,2)) AS net_amount
        FROM {catalog}.silver.order_items oi
        LEFT JOIN {catalog}.silver.orders o
               ON oi.order_id = o.order_id
        LEFT JOIN {catalog}.gold.dim_date dd
               ON o.order_date = dd.calendar_date
        LEFT JOIN {catalog}.gold.dim_customer dc
               ON o.customer_id = dc.customer_id
              AND o.order_date >= dc.effective_from
              AND (dc.effective_to IS NULL OR o.order_date < dc.effective_to)
    ) AS s
    ON t.order_id = s.order_id AND t.line_number = s.line_number
    WHEN MATCHED     THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

# ============================================================
# 3. Validation
# ============================================================

# # --- Row count comparison ---
# print("=== Row counts ===")
# display(spark.sql(f"""
#     SELECT 'silver.orders'      AS table_name, COUNT(*) AS row_count FROM {catalog}.silver.orders
#     UNION ALL SELECT 'fact_orders',     COUNT(*) FROM {catalog}.gold.fact_orders
#     UNION ALL SELECT 'silver.order_items', COUNT(*) FROM {catalog}.silver.order_items
#     UNION ALL SELECT 'fact_order_line', COUNT(*) FROM {catalog}.gold.fact_order_line
#     ORDER BY table_name
# """))

# # --- Every order has exactly one fact-order record ---
# print("=== Duplicate / missing order check (should be 0) ===")
# display(spark.sql(f"""
#     SELECT COUNT(*) AS issue_count
#     FROM (
#         SELECT order_id
#         FROM {catalog}.gold.fact_orders
#         GROUP BY order_id
#         HAVING COUNT(*) <> 1
#     )
# """))

# # --- Totals reconciliation: fact_orders vs fact_order_line ---
# print("=== Net amount mismatch check (should be 0) ===")
# display(spark.sql(f"""
#     SELECT COUNT(*) AS mismatch_count
#     FROM (
#         SELECT fo.order_id
#         FROM {catalog}.gold.fact_orders fo
#         JOIN {catalog}.gold.fact_order_line fl
#           ON fo.order_id = fl.order_id
#         GROUP BY fo.order_id, fo.net_item_amount
#         HAVING fo.net_item_amount <> SUM(fl.net_amount)
#     )
# """))

# # --- Sample: fact_orders ---
# print("=== Sample: fact_orders ===")
# display(spark.sql(f"SELECT * FROM {catalog}.gold.fact_orders ORDER BY order_id LIMIT 10"))

# # --- Sample: fact_order_line ---
# print("=== Sample: fact_order_line ===")
# display(spark.sql(f"SELECT * FROM {catalog}.gold.fact_order_line ORDER BY order_id, line_number LIMIT 10"))

In [0]:
# --- gold.agg_daily_store_sales: one row per store per date (zero-filled for no-activity days) --

spark.sql(f"""
    CREATE OR REPLACE TABLE {catalog}.gold.agg_daily_store_sales AS
    WITH order_dates AS (
        SELECT MIN(dd.calendar_date) AS min_date,
               MAX(dd.calendar_date) AS max_date
        FROM {catalog}.gold.fact_orders fo
        JOIN {catalog}.gold.dim_date dd ON fo.order_date_key = dd.date_key
    ),
    date_spine AS (
        SELECT explode(sequence(min_date, max_date, INTERVAL 1 DAY)) AS order_date
        FROM order_dates
    ),
    store_dates AS (
        SELECT ds.order_date, st.store_id, st.city
        FROM date_spine ds
        CROSS JOIN {catalog}.gold.dim_store st
    ),
    store_agg AS (
        SELECT
            dd.calendar_date AS order_date,
            fo.store_id,
            CAST(SUM(CASE WHEN fo.order_status = 'DELIVERED'               THEN 1 ELSE 0 END) AS INT) AS delivered_orders,
            CAST(SUM(CASE WHEN UPPER(fo.order_status) LIKE 'CANCEL%'     THEN 1 ELSE 0 END) AS INT) AS canceled_orders,
            CAST(SUM(fo.units) AS INT)                                                       AS units_sold,
            CAST(SUM(fo.gross_amount)    AS DECIMAL(12,2))                                  AS gross_amount,
            CAST(SUM(fo.discount_amount) AS DECIMAL(12,2))                                  AS discount_amount,
            CAST(SUM(fo.net_item_amount) AS DECIMAL(12,2))                                  AS net_item_revenue,
            CAST(SUM(fo.delivery_fee)    AS DECIMAL(10,2))                                  AS delivery_fee_revenue,
            CAST(SUM(CASE WHEN fo.is_on_time = TRUE                          THEN 1 ELSE 0 END) AS INT) AS on_time_orders
        FROM {catalog}.gold.fact_orders fo
        JOIN {catalog}.gold.dim_date dd ON fo.order_date_key = dd.date_key
        GROUP BY dd.calendar_date, fo.store_id
    )
    SELECT
        sd.order_date,
        sd.store_id,
        sd.city,
        COALESCE(sa.delivered_orders, 0)       AS delivered_orders,
        COALESCE(sa.canceled_orders, 0)        AS canceled_orders,
        COALESCE(sa.units_sold, 0)             AS units_sold,
        COALESCE(sa.gross_amount, 0)            AS gross_amount,
        COALESCE(sa.discount_amount, 0)        AS discount_amount,
        COALESCE(sa.net_item_revenue, 0)       AS net_item_revenue,
        COALESCE(sa.delivery_fee_revenue, 0)   AS delivery_fee_revenue,
        COALESCE(sa.on_time_orders, 0)          AS on_time_orders,
        CASE
            WHEN COALESCE(sa.delivered_orders, 0) = 0 THEN NULL
            ELSE CAST(100.0 * sa.on_time_orders / sa.delivered_orders AS DECIMAL(5,2))
        END AS on_time_rate_pct
    FROM store_dates sd
    LEFT JOIN store_agg sa
        ON sd.order_date = sa.order_date
       AND sd.store_id  = sa.store_id
""")

# # --- Validation ---
# print("=== Row count ===")
# display(spark.sql(f"SELECT COUNT(*) AS row_count FROM {catalog}.gold.agg_daily_store_sales"))

# print("=== Sample (first 10 rows) ===")
# display(spark.sql(f"""
#     SELECT * FROM {catalog}.gold.agg_daily_store_sales
#     ORDER BY order_date, store_id
#     LIMIT 10
# """))

# print("=== Zero-activity store-days (no orders at all) ===")
# display(spark.sql(f"""
#     SELECT COUNT(*) AS zero_activity_days
#     FROM {catalog}.gold.agg_daily_store_sales
#     WHERE delivered_orders = 0 AND canceled_orders = 0
# """))

In [0]:
# --- gold.v_kpi_summary: single-row KPI rollup for the whole period ---

spark.sql(f"""
    CREATE OR REPLACE VIEW {catalog}.gold.v_kpi_summary AS
    WITH order_stats AS (
        SELECT
            CAST(SUM(CASE WHEN order_status = 'DELIVERED'                 THEN 1 ELSE 0 END) AS INT) AS delivered_orders,
            CAST(SUM(CASE WHEN UPPER(order_status) LIKE 'CANCEL%'         THEN 1 ELSE 0 END) AS INT) AS cancelled_orders,
            CAST(SUM(net_item_amount) AS DECIMAL(12,2))                                              AS net_item_revenue,
            CAST(SUM(CASE WHEN is_on_time = TRUE                          THEN 1 ELSE 0 END) AS INT) AS on_time_orders
        FROM {catalog}.gold.fact_orders
    ),
    customer_deliveries AS (
        SELECT customer_id, COUNT(*) AS delivered_count
        FROM {catalog}.gold.fact_orders
        WHERE order_status = 'DELIVERED'
        GROUP BY customer_id
    ),
    customer_agg AS (
        SELECT
            COUNT(*)                                               AS active_customers,
            SUM(CASE WHEN delivered_count >= 2 THEN 1 ELSE 0 END) AS repeat_customers
        FROM customer_deliveries
    )
    SELECT
        os.delivered_orders,
        os.cancelled_orders,
        os.net_item_revenue,
        CASE
            WHEN os.delivered_orders = 0 THEN NULL
            ELSE CAST(os.net_item_revenue / os.delivered_orders AS DECIMAL(12,2))
        END AS avg_order_value,
        CASE
            WHEN os.delivered_orders = 0 THEN NULL
            ELSE CAST(100.0 * os.on_time_orders / os.delivered_orders AS DECIMAL(5,2))
        END AS on_time_rate_pct,
        CASE
            WHEN (os.delivered_orders + os.cancelled_orders) = 0 THEN NULL
            ELSE CAST(100.0 * os.cancelled_orders / (os.delivered_orders + os.cancelled_orders) AS DECIMAL(5,2))
        END AS cancellation_rate_pct,
        CAST(ca.active_customers AS INT) AS active_customers,
        CASE
            WHEN ca.active_customers = 0 THEN NULL
            ELSE CAST(100.0 * ca.repeat_customers / ca.active_customers AS DECIMAL(5,2))
        END AS repeat_customer_rate_pct
    FROM order_stats os
    CROSS JOIN customer_agg ca
""")

# --- Preview ---
# display(spark.sql(f"SELECT * FROM {catalog}.gold.v_kpi_summary"))

In [0]:
# --- ops.reconciliation_results: cross-layer data quality checks (append-only) ---

# Schema already created in 00_set_up, but guard just in case
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.ops")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.ops.reconciliation_results (
        run_ts        TIMESTAMP,
        check_name    STRING,
        source_value  DECIMAL(18,2),
        target_value  DECIMAL(18,2),
        status        STRING
    )
""")

# Single INSERT → all 5 rows share one CURRENT_TIMESTAMP() value
spark.sql(f"""
    INSERT INTO {catalog}.ops.reconciliation_results
    SELECT
        CURRENT_TIMESTAMP()                              AS run_ts,
        t.check_name,
        t.source_value,
        t.target_value,
        CASE WHEN t.source_value = t.target_value
             THEN 'PASS' ELSE 'FAIL' END                 AS status
    FROM (
        -- 1. ORDER_COUNT: distinct order_id in bronze vs silver.orders + quarantine
        SELECT
            'ORDER_COUNT' AS check_name,
            CAST((SELECT COUNT(DISTINCT order_id) FROM {catalog}.bronze.orders) AS DECIMAL(18,2)) AS source_value,
            CAST((SELECT COUNT(*) FROM {catalog}.silver.orders) +
                 (SELECT COUNT(*) FROM {catalog}.silver.quarantine_orders)      AS DECIMAL(18,2)) AS target_value

        UNION ALL

        -- 2. LINE_COUNT: silver.order_items vs gold.fact_order_line
        SELECT
            'LINE_COUNT',
            CAST((SELECT COUNT(*) FROM {catalog}.silver.order_items)   AS DECIMAL(18,2)),
            CAST((SELECT COUNT(*) FROM {catalog}.gold.fact_order_line)  AS DECIMAL(18,2))

        UNION ALL

        -- 3. NET_REVENUE: silver order_items net_amount (DELIVERED only) vs gold agg net_item_revenue
        SELECT
            'NET_REVENUE',
            CAST((SELECT COALESCE(SUM(oi.net_amount), 0)
                  FROM {catalog}.silver.order_items oi
                  JOIN {catalog}.silver.orders o ON oi.order_id = o.order_id
                  WHERE o.order_status = 'DELIVERED')                         AS DECIMAL(18,2)),
            CAST((SELECT COALESCE(SUM(net_item_revenue), 0)
                  FROM {catalog}.gold.agg_daily_store_sales)                   AS DECIMAL(18,2))

        UNION ALL

        -- 4. DELIVERED_ORDERS: gold.fact_orders count vs gold.agg_daily_store_sales sum
        SELECT
            'DELIVERED_ORDERS',
            CAST((SELECT COUNT(*) FROM {catalog}.gold.fact_orders
                  WHERE order_status = 'DELIVERED')                            AS DECIMAL(18,2)),
            CAST((SELECT COALESCE(SUM(delivered_orders), 0)
                  FROM {catalog}.gold.agg_daily_store_sales)                   AS DECIMAL(18,2))

        UNION ALL

        -- 5. CURRENT_CUSTOMERS: distinct customer_id in silver vs current rows in gold.dim_customer
        SELECT
            'CURRENT_CUSTOMERS',
            CAST((SELECT COUNT(DISTINCT customer_id) FROM {catalog}.silver.customers) AS DECIMAL(18,2)),
            CAST((SELECT COUNT(*) FROM {catalog}.gold.dim_customer WHERE is_current = TRUE) AS DECIMAL(18,2))
    ) t
""")

# --- Preview latest run ---
display(spark.sql(f"""
    SELECT *
    FROM {catalog}.ops.reconciliation_results
    ORDER BY run_ts DESC, check_name
    LIMIT 5
"""))